# VAE + DBSCAN for PSD Signature Detection

## Problem Statement

We have Power Spectral Density (PSD) plots collected over time:
- Each PSD: 14,000 frequency points × 14,000 power values
- ~8000 PSDs collected over a week (one per timestamp)
- **Goal**: Identify recurring patterns ("signatures") without prior knowledge
- **Goal**: Distinguish normal behavior from anomalies

## Approach

1. Generate synthetic PSD data with known patterns
2. Apply **Variational Autoencoder (VAE)** for non-linear dimensionality reduction
3. Use DBSCAN for clustering in the latent space
4. Visualize clusters and representative traces

## Comparison to PCA

Unlike PCA (linear dimensionality reduction), VAE can:
- Capture **non-linear** relationships in the data
- Learn a **probabilistic** latent representation
- Act as a powerful **denoising** mechanism
- Generate new samples from the learned distribution

## Examples
1. Clean synthetic data with distinct signatures
2. Perturbed data with noise to test robustness

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import seaborn as sns
from typing import Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data Generation Functions

Reusing the same synthetic data generation as in the PCA notebook for fair comparison.

In [ ]:
def generate_synthetic_psd_signatures(n_frequencies: int = 14000) -> Tuple[np.ndarray, List[callable]]:
    """
    Generate base signature patterns for synthetic PSD data.
    """
    frequencies = np.linspace(0, 6000, n_frequencies)  # 0 to 6 GHz
    
    def signature_1(freqs):
        """Flat baseline with narrow peak at 1 GHz"""
        baseline = -80 * np.ones_like(freqs)
        peak = 30 * np.exp(-((freqs - 1000)**2) / (50**2))
        return baseline + peak
    
    def signature_2(freqs):
        """Flat baseline with two peaks at 2 GHz and 4 GHz"""
        baseline = -80 * np.ones_like(freqs)
        peak1 = 25 * np.exp(-((freqs - 2000)**2) / (80**2))
        peak2 = 25 * np.exp(-((freqs - 4000)**2) / (80**2))
        return baseline + peak1 + peak2
    
    def signature_3(freqs):
        """Sloped baseline with broad peak at 3 GHz"""
        baseline = -85 + (freqs / 6000) * 10
        peak = 35 * np.exp(-((freqs - 3000)**2) / (200**2))
        return baseline + peak
    
    def signature_4(freqs):
        """Multiple small peaks (harmonics)"""
        baseline = -80 * np.ones_like(freqs)
        harmonics = np.zeros_like(freqs)
        for i, f_center in enumerate([1000, 2000, 3000, 4000], 1):
            harmonics += (30 / i) * np.exp(-((freqs - f_center)**2) / (40**2))
        return baseline + harmonics
    
    def signature_5(freqs):
        """Wide-band elevated region"""
        baseline = -80 * np.ones_like(freqs)
        elevated = 20 * (np.tanh((freqs - 2000) / 200) - np.tanh((freqs - 4000) / 200))
        return baseline + elevated
    
    return frequencies, [signature_1, signature_2, signature_3, signature_4, signature_5]


def generate_psd_dataset(n_samples: int, 
                        n_frequencies: int = 14000,
                        noise_level: float = 0.0,
                        signature_distribution: List[float] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate a synthetic dataset of PSD measurements.
    """
    frequencies, signature_funcs = generate_synthetic_psd_signatures(n_frequencies)
    n_signatures = len(signature_funcs)
    
    if signature_distribution is None:
        signature_distribution = np.ones(n_signatures) / n_signatures
    
    psd_data = np.zeros((n_samples, n_frequencies))
    labels = np.random.choice(n_signatures, size=n_samples, p=signature_distribution)
    
    for i in range(n_samples):
        sig_func = signature_funcs[labels[i]]
        psd_data[i] = sig_func(frequencies)
        
        if noise_level > 0:
            psd_data[i] += np.random.normal(0, noise_level, n_frequencies)
    
    return psd_data, frequencies, labels

## VAE Architecture

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder for PSD data.
    
    Architecture:
    - Encoder: 14000 -> 1024 -> 256 -> latent_dim * 2 (mean and log_var)
    - Decoder: latent_dim -> 256 -> 1024 -> 14000
    """
    
    def __init__(self, input_dim: int = 14000, latent_dim: int = 10, hidden_dims: List[int] = [1024, 256]):
        super(VAE, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dims = hidden_dims
        
        # Encoder layers
        encoder_layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent space: mean and log variance
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)
        
        # Decoder layers
        decoder_layers = []
        prev_dim = latent_dim
        
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = h_dim
        
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        """Encode input to latent space parameters."""
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + sigma * epsilon."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Decode latent vector to reconstruction."""
        return self.decoder(z)
    
    def forward(self, x):
        """Forward pass through VAE."""
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar, z


def vae_loss_function(recon_x, x, mu, logvar, beta: float = 1.0):
    """
    VAE loss = Reconstruction loss + KL divergence.
    
    Args:
        recon_x: Reconstructed input
        x: Original input
        mu: Mean of latent distribution
        logvar: Log variance of latent distribution
        beta: Weight for KL divergence (beta-VAE)
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_div = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kl_div, recon_loss, kl_div

## Training Functions

In [ ]:
def train_vae(vae: VAE, 
              train_loader: DataLoader,
              n_epochs: int = 50,
              learning_rate: float = 1e-3,
              beta: float = 1.0,
              verbose: bool = True) -> Dict:
    """
    Train the VAE model.
    
    Args:
        vae: VAE model
        train_loader: DataLoader for training data
        n_epochs: Number of training epochs
        learning_rate: Learning rate
        beta: Weight for KL divergence (beta-VAE)
        verbose: Print training progress
        
    Returns:
        Dictionary with training history
    """
    optimizer = optim.Adam(vae.parameters(), lr=learning_rate)
    
    history = {
        'total_loss': [],
        'recon_loss': [],
        'kl_loss': []
    }
    
    vae.train()
    
    for epoch in range(n_epochs):
        epoch_total_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        
        for batch_data in train_loader:
            if isinstance(batch_data, list):
                data = batch_data[0].to(device)
            else:
                data = batch_data.to(device)
            
            optimizer.zero_grad()
            
            recon_batch, mu, logvar, _ = vae(data)
            loss, recon_loss, kl_loss = vae_loss_function(recon_batch, data, mu, logvar, beta)
            
            loss.backward()
            optimizer.step()
            
            epoch_total_loss += loss.item()
            epoch_recon_loss += recon_loss.item()
            epoch_kl_loss += kl_loss.item()
        
        # Average over dataset
        n_samples = len(train_loader.dataset)
        avg_total_loss = epoch_total_loss / n_samples
        avg_recon_loss = epoch_recon_loss / n_samples
        avg_kl_loss = epoch_kl_loss / n_samples
        
        history['total_loss'].append(avg_total_loss)
        history['recon_loss'].append(avg_recon_loss)
        history['kl_loss'].append(avg_kl_loss)
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{n_epochs}], '
                  f'Total Loss: {avg_total_loss:.4f}, '
                  f'Recon Loss: {avg_recon_loss:.4f}, '
                  f'KL Loss: {avg_kl_loss:.4f}')
    
    return history


def extract_latent_features(vae: VAE, data: np.ndarray, batch_size: int = 128) -> np.ndarray:
    """
    Extract latent features from data using trained VAE encoder.
    
    Args:
        vae: Trained VAE model
        data: Input data (numpy array)
        batch_size: Batch size for processing
        
    Returns:
        Latent features (numpy array)
    """
    vae.eval()
    
    dataset = TensorDataset(torch.FloatTensor(data))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    latent_features = []
    
    with torch.no_grad():
        for batch_data in loader:
            if isinstance(batch_data, list):
                batch = batch_data[0].to(device)
            else:
                batch = batch_data.to(device)
            
            mu, _ = vae.encode(batch)
            latent_features.append(mu.cpu().numpy())
    
    return np.vstack(latent_features)


def reconstruct_data(vae: VAE, data: np.ndarray, batch_size: int = 128) -> np.ndarray:
    """
    Reconstruct data using trained VAE.
    """
    vae.eval()
    
    dataset = TensorDataset(torch.FloatTensor(data))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    reconstructions = []
    
    with torch.no_grad():
        for batch_data in loader:
            if isinstance(batch_data, list):
                batch = batch_data[0].to(device)
            else:
                batch = batch_data.to(device)
            
            recon, _, _, _ = vae(batch)
            reconstructions.append(recon.cpu().numpy())
    
    return np.vstack(reconstructions)

## Visualization Functions

In [ ]:
def plot_training_history(history: Dict):
    """
    Plot VAE training history.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(history['total_loss'])
    axes[0].set_title('Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(history['recon_loss'])
    axes[1].set_title('Reconstruction Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(history['kl_loss'])
    axes[2].set_title('KL Divergence')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_latent_space_2d(latent_features: np.ndarray,
                        cluster_labels: np.ndarray,
                        true_labels: np.ndarray = None,
                        title: str = "DBSCAN Clusters in VAE Latent Space"):
    """
    Plot clusters in 2D latent space (first two dimensions).
    """
    fig, axes = plt.subplots(1, 2 if true_labels is not None else 1, 
                            figsize=(14 if true_labels is not None else 8, 5))
    if true_labels is None:
        axes = [axes]
    
    # Plot DBSCAN clusters
    unique_clusters = set(cluster_labels)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_clusters)))
    
    for cluster_id, color in zip(sorted(unique_clusters), colors):
        mask = cluster_labels == cluster_id
        label = f'Noise' if cluster_id == -1 else f'Cluster {cluster_id}'
        marker = 'x' if cluster_id == -1 else 'o'
        alpha = 0.3 if cluster_id == -1 else 0.6
        
        axes[0].scatter(latent_features[mask, 0], latent_features[mask, 1],
                       c=[color], label=label, marker=marker, alpha=alpha, s=50)
    
    axes[0].set_xlabel('Latent Dim 1')
    axes[0].set_ylabel('Latent Dim 2')
    axes[0].set_title(title)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot true labels if provided
    if true_labels is not None:
        unique_true = set(true_labels)
        colors_true = plt.cm.rainbow(np.linspace(0, 1, len(unique_true)))
        
        for true_label, color in zip(sorted(unique_true), colors_true):
            mask = true_labels == true_label
            axes[1].scatter(latent_features[mask, 0], latent_features[mask, 1],
                          c=[color], label=f'Signature {true_label}', alpha=0.6, s=50)
        
        axes[1].set_xlabel('Latent Dim 1')
        axes[1].set_ylabel('Latent Dim 2')
        axes[1].set_title('True Signature Labels')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_reconstructions(original_data: np.ndarray,
                        reconstructed_data: np.ndarray,
                        frequencies: np.ndarray,
                        n_samples: int = 3,
                        title_prefix: str = ""):
    """
    Plot original vs VAE-reconstructed PSDs.
    """
    n_samples = min(n_samples, len(original_data))
    sample_indices = np.random.choice(len(original_data), n_samples, replace=False)
    
    fig, axes = plt.subplots(n_samples, 1, figsize=(14, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]
    
    for idx, sample_idx in enumerate(sample_indices):
        ax = axes[idx]
        
        # Plot original and reconstructed
        ax.plot(frequencies, original_data[sample_idx], 'b-', alpha=0.7, linewidth=2, label='Original')
        ax.plot(frequencies, reconstructed_data[sample_idx], 'r--', alpha=0.7, linewidth=2, label='VAE Reconstruction')
        
        # Plot difference
        ax_twin = ax.twinx()
        error = original_data[sample_idx] - reconstructed_data[sample_idx]
        ax_twin.fill_between(frequencies, error, alpha=0.3, color='gray', label='Reconstruction Error')
        ax_twin.set_ylabel('Reconstruction Error (dBm)', color='gray')
        ax_twin.tick_params(axis='y', labelcolor='gray')
        
        # Compute R²
        ss_res = np.sum(error ** 2)
        ss_tot = np.sum((original_data[sample_idx] - np.mean(original_data[sample_idx])) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        
        ax.set_title(f'{title_prefix}Sample {sample_idx} (R² = {r2:.4f})')
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dBm)')
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_representative_traces(psd_data: np.ndarray,
                               frequencies: np.ndarray,
                               cluster_labels: np.ndarray,
                               n_examples: int = 3):
    """
    Plot representative PSD traces from each cluster.
    """
    unique_clusters = sorted(set(cluster_labels))
    n_clusters = len(unique_clusters)
    
    fig, axes = plt.subplots(n_clusters, 1, figsize=(14, 4 * n_clusters))
    if n_clusters == 1:
        axes = [axes]
    
    for idx, cluster_id in enumerate(unique_clusters):
        ax = axes[idx]
        mask = cluster_labels == cluster_id
        cluster_data = psd_data[mask]
        
        n_to_plot = min(n_examples, len(cluster_data))
        indices = np.random.choice(len(cluster_data), n_to_plot, replace=False)
        
        for i in indices:
            ax.plot(frequencies, cluster_data[i], alpha=0.4, linewidth=1)
        
        mean_trace = np.mean(cluster_data, axis=0)
        ax.plot(frequencies, mean_trace, 'k-', linewidth=2, label='Mean')
        
        std_trace = np.std(cluster_data, axis=0)
        ax.fill_between(frequencies, mean_trace - std_trace, mean_trace + std_trace,
                        alpha=0.2, color='gray', label='±1 std')
        
        label = f'Noise/Outliers (n={len(cluster_data)})' if cluster_id == -1 else f'Cluster {cluster_id} (n={len(cluster_data)})'
        ax.set_title(label)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dBm)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
# Example 1: Clean Synthetic Data

Train VAE on clean synthetic PSD data and apply DBSCAN clustering.

## Generate Clean Dataset

In [ ]:
# Parameters
N_SAMPLES_CLEAN = 1000
N_FREQUENCIES = 14000
NOISE_LEVEL_CLEAN = 0.0
LATENT_DIM = 10
BATCH_SIZE = 64
N_EPOCHS = 50

# Generate dataset
print("Generating clean synthetic PSD dataset...")
psd_data_clean, frequencies, true_labels_clean = generate_psd_dataset(
    n_samples=N_SAMPLES_CLEAN,
    n_frequencies=N_FREQUENCIES,
    noise_level=NOISE_LEVEL_CLEAN
)

print(f"\nDataset shape: {psd_data_clean.shape}")
print(f"Frequency range: {frequencies[0]:.1f} - {frequencies[-1]:.1f} MHz")
print(f"Number of unique signatures: {len(set(true_labels_clean))}")

## Standardize Data and Create DataLoader

In [ ]:
# Standardize the data
scaler_clean = StandardScaler()
psd_data_clean_scaled = scaler_clean.fit_transform(psd_data_clean)

# Create DataLoader
train_dataset_clean = TensorDataset(torch.FloatTensor(psd_data_clean_scaled))
train_loader_clean = DataLoader(train_dataset_clean, batch_size=BATCH_SIZE, shuffle=True)

print(f"Created DataLoader with {len(train_dataset_clean)} samples, batch size {BATCH_SIZE}")

## Train VAE

In [ ]:
# Initialize VAE
vae_clean = VAE(input_dim=N_FREQUENCIES, latent_dim=LATENT_DIM, hidden_dims=[1024, 256]).to(device)

print(f"VAE Architecture:")
print(f"  Input dimension: {N_FREQUENCIES}")
print(f"  Hidden dimensions: [1024, 256]")
print(f"  Latent dimension: {LATENT_DIM}")
print(f"  Total parameters: {sum(p.numel() for p in vae_clean.parameters()):,}")
print(f"\nTraining VAE on clean data...")

# Train VAE
history_clean = train_vae(
    vae_clean,
    train_loader_clean,
    n_epochs=N_EPOCHS,
    learning_rate=1e-3,
    beta=1.0,
    verbose=True
)

## Visualize Training History

In [ ]:
plot_training_history(history_clean)

## Extract Latent Features

In [ ]:
# Extract latent features
latent_features_clean = extract_latent_features(vae_clean, psd_data_clean_scaled, batch_size=BATCH_SIZE)

print(f"Latent features shape: {latent_features_clean.shape}")
print(f"Reduced from {N_FREQUENCIES} to {LATENT_DIM} dimensions")

## Apply DBSCAN Clustering

In [ ]:
# Apply DBSCAN
EPS_CLEAN = 2.0
MIN_SAMPLES_CLEAN = 10

dbscan_clean = DBSCAN(eps=EPS_CLEAN, min_samples=MIN_SAMPLES_CLEAN)
cluster_labels_clean = dbscan_clean.fit_predict(latent_features_clean)

n_clusters = len(set(cluster_labels_clean)) - (1 if -1 in cluster_labels_clean else 0)
n_noise = list(cluster_labels_clean).count(-1)

print(f"\nDBSCAN clustering completed:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Number of noise points: {n_noise}")
if n_clusters > 0:
    print(f"  Cluster sizes: {np.bincount(cluster_labels_clean[cluster_labels_clean >= 0])}")

## Visualize Clusters in Latent Space

In [ ]:
plot_latent_space_2d(latent_features_clean, cluster_labels_clean, true_labels_clean,
                    title="DBSCAN Clusters (Clean Data - VAE Latent Space)")

## Plot Representative Traces

In [ ]:
plot_representative_traces(psd_data_clean, frequencies, cluster_labels_clean, n_examples=5)

## VAE Reconstruction Quality

In [ ]:
# Reconstruct data
reconstructed_clean = reconstruct_data(vae_clean, psd_data_clean_scaled, batch_size=BATCH_SIZE)
reconstructed_clean = scaler_clean.inverse_transform(reconstructed_clean)

# Plot reconstructions
plot_reconstructions(psd_data_clean, reconstructed_clean, frequencies, n_samples=3, title_prefix="Clean Data - ")

## Cluster Quality Metrics

In [ ]:
# Compare cluster assignments to true labels
non_noise_mask_clean = cluster_labels_clean >= 0

if non_noise_mask_clean.sum() > 0:
    ari_clean = adjusted_rand_score(true_labels_clean[non_noise_mask_clean], 
                                    cluster_labels_clean[non_noise_mask_clean])
    nmi_clean = normalized_mutual_info_score(true_labels_clean[non_noise_mask_clean], 
                                             cluster_labels_clean[non_noise_mask_clean])
    
    print("Clustering Quality Metrics (excluding noise points):")
    print(f"  Adjusted Rand Index: {ari_clean:.4f} (1.0 = perfect match)")
    print(f"  Normalized Mutual Information: {nmi_clean:.4f} (1.0 = perfect match)")
else:
    print("All points classified as noise - try adjusting DBSCAN parameters")

---
# Example 2: Noisy Synthetic Data

Train VAE on noisy synthetic PSD data to test robustness.

## Generate Noisy Dataset

In [ ]:
# Parameters
N_SAMPLES_NOISY = 1000
NOISE_LEVEL_NOISY = 2.0  # 2 dB standard deviation

# Generate dataset
print("Generating noisy synthetic PSD dataset...")
psd_data_noisy, _, true_labels_noisy = generate_psd_dataset(
    n_samples=N_SAMPLES_NOISY,
    n_frequencies=N_FREQUENCIES,
    noise_level=NOISE_LEVEL_NOISY
)

print(f"\nDataset shape: {psd_data_noisy.shape}")
print(f"Noise level: {NOISE_LEVEL_NOISY} dB")
print(f"Number of unique signatures: {len(set(true_labels_noisy))}")

## Standardize Data and Create DataLoader

In [ ]:
# Standardize the data
scaler_noisy = StandardScaler()
psd_data_noisy_scaled = scaler_noisy.fit_transform(psd_data_noisy)

# Create DataLoader
train_dataset_noisy = TensorDataset(torch.FloatTensor(psd_data_noisy_scaled))
train_loader_noisy = DataLoader(train_dataset_noisy, batch_size=BATCH_SIZE, shuffle=True)

print(f"Created DataLoader with {len(train_dataset_noisy)} samples, batch size {BATCH_SIZE}")

## Train VAE

In [ ]:
# Initialize VAE
vae_noisy = VAE(input_dim=N_FREQUENCIES, latent_dim=LATENT_DIM, hidden_dims=[1024, 256]).to(device)

print(f"Training VAE on noisy data...")

# Train VAE
history_noisy = train_vae(
    vae_noisy,
    train_loader_noisy,
    n_epochs=N_EPOCHS,
    learning_rate=1e-3,
    beta=1.0,
    verbose=True
)

## Visualize Training History

In [ ]:
plot_training_history(history_noisy)

## Extract Latent Features

In [ ]:
# Extract latent features
latent_features_noisy = extract_latent_features(vae_noisy, psd_data_noisy_scaled, batch_size=BATCH_SIZE)

print(f"Latent features shape: {latent_features_noisy.shape}")

## Apply DBSCAN Clustering

In [ ]:
# Apply DBSCAN
EPS_NOISY = 2.5  # May need adjustment
MIN_SAMPLES_NOISY = 10

dbscan_noisy = DBSCAN(eps=EPS_NOISY, min_samples=MIN_SAMPLES_NOISY)
cluster_labels_noisy = dbscan_noisy.fit_predict(latent_features_noisy)

n_clusters_noisy = len(set(cluster_labels_noisy)) - (1 if -1 in cluster_labels_noisy else 0)
n_noise_noisy = list(cluster_labels_noisy).count(-1)

print(f"\nDBSCAN clustering completed:")
print(f"  Number of clusters: {n_clusters_noisy}")
print(f"  Number of noise points: {n_noise_noisy}")
if n_clusters_noisy > 0:
    print(f"  Cluster sizes: {np.bincount(cluster_labels_noisy[cluster_labels_noisy >= 0])}")

## Visualize Clusters in Latent Space

In [ ]:
plot_latent_space_2d(latent_features_noisy, cluster_labels_noisy, true_labels_noisy,
                    title="DBSCAN Clusters (Noisy Data - VAE Latent Space)")

## Plot Representative Traces

In [ ]:
plot_representative_traces(psd_data_noisy, frequencies, cluster_labels_noisy, n_examples=5)

## VAE Reconstruction Quality (Denoising Effect)

In [ ]:
# Reconstruct data
reconstructed_noisy = reconstruct_data(vae_noisy, psd_data_noisy_scaled, batch_size=BATCH_SIZE)
reconstructed_noisy = scaler_noisy.inverse_transform(reconstructed_noisy)

# Plot reconstructions
print("Notice how the VAE reconstruction removes noise while preserving signal structure:")
plot_reconstructions(psd_data_noisy, reconstructed_noisy, frequencies, n_samples=3, title_prefix="Noisy Data - ")

## Cluster Quality Metrics

In [ ]:
# Compare cluster assignments to true labels
non_noise_mask_noisy = cluster_labels_noisy >= 0

if non_noise_mask_noisy.sum() > 0:
    ari_noisy = adjusted_rand_score(true_labels_noisy[non_noise_mask_noisy], 
                                    cluster_labels_noisy[non_noise_mask_noisy])
    nmi_noisy = normalized_mutual_info_score(true_labels_noisy[non_noise_mask_noisy], 
                                             cluster_labels_noisy[non_noise_mask_noisy])
    
    print("Clustering Quality Metrics (excluding noise points):")
    print(f"  Adjusted Rand Index: {ari_noisy:.4f} (1.0 = perfect match)")
    print(f"  Normalized Mutual Information: {nmi_noisy:.4f} (1.0 = perfect match)")
    
    print("\nComparison to Clean Data:")
    if 'ari_clean' in locals():
        print(f"  ARI degradation: {ari_clean - ari_noisy:.4f}")
        print(f"  NMI degradation: {nmi_clean - nmi_noisy:.4f}")
else:
    print("All points classified as noise - try adjusting DBSCAN parameters")

---
# Summary and Comparison

## VAE Advantages

1. **Non-linear Dimensionality Reduction**: VAE can capture complex, non-linear relationships that PCA cannot
2. **Denoising Capability**: VAE learns to reconstruct clean signals from noisy inputs
3. **Probabilistic Representation**: The latent space has a probabilistic interpretation
4. **Generative Model**: Can generate new synthetic PSD samples

## VAE vs PCA Trade-offs

**VAE Pros:**
- Better for complex, non-linear patterns
- Superior denoising through learned reconstruction
- Can model more sophisticated data distributions

**VAE Cons:**
- Requires training (computational cost)
- More hyperparameters to tune
- Non-deterministic (initialization matters)
- May overfit with small datasets

**PCA Pros:**
- Fast, deterministic, no training required
- Fewer hyperparameters
- Works well for linear relationships
- Guaranteed global optimum

**PCA Cons:**
- Limited to linear transformations
- May miss complex patterns

## When to Use Which?

- **Use PCA**: Quick exploration, linear patterns, small datasets, need interpretability
- **Use VAE**: Complex patterns, need denoising, large datasets, willing to invest in training
- **Use Both**: Compare results to understand data structure better!